# Exp29 — Direction-Transfer Test (DAS): Gender vs Word Order

Fresh, self-contained follow-up to Exp27 (word-order whole-state patching, verdict
INVERSION_ABANDONED) and Exp28 (gender whole-state patching, verdict INSTRUMENT_INADEQUATE).
See the handoff doc for full background; the short version:

- **Why not whole-state patching again:** it overwrites language identity, content, and the
  feature of interest at once, so a null result can mean "readout is language-specific" rather
  than "code isn't shared," and a positive result can be non-specific disruption rather than
  real transfer. This notebook instead learns a single **additive direction** per language/
  construction and tests whether adding it elsewhere reproduces the effect (sufficiency) and
  whether removing it degrades the source language's own behavior (necessity).
- **Why DAS, not diff-of-means:** a gradient-/search-optimized 1D subspace is a strictly
  stronger standard than a mean-difference direction (precedent: Boguraev et al. 2026,
  arXiv:2608.28924, use Distributed Alignment Search for exactly this kind of cross-lingual
  transfer claim). We approximate DAS with **SPSA** (simultaneous perturbation stochastic
  approximation) rather than reverse-mode autograd: SPSA needs only 2 forward passes per
  update regardless of dimensionality, which (a) reuses the existing forward-hook-only infra
  with no need to keep a differentiable graph alive through the rest of a 9B/27B model, and
  (b) runs identically on the toy test harness and the real model, so the same code is
  structurally tested offline. This is a deliberate deviation from Boguraev et al.'s literal
  method — flag it as such in any writeup, and treat it as a weaker-but-compatible estimator
  of the same causal subspace, not an identical reproduction.
- **Why necessity + a cue-only falsifier, not just sufficiency:** per the triangulation
  standard (Long, "Triangulation law" — already in the project's reading), causal mechanism
  claims need necessity (ablating degrades the target behavior), sufficiency (patching
  transfers it), and invariance (cue-only falsifiers do NOT transfer it). Exp27/28 and the
  original Exp29 proposal only tested sufficiency. This version gates every TRANSFER
  classification on sufficiency AND necessity both passing, and adds a real language-identity
  direction (not just random/permutation noise) as an explicit cue-only falsifier.

**What this notebook does NOT establish:** a transferring direction shows an additive,
language-portable *code* exists at a readout — not where or how it was computed, nor that the
same circuit (same heads/MLPs) produces it in each language. That is a separate, more
expensive circuit-localization question (flagged as a candidate Exp30, not attempted here).

**Data status:** the item-bank cell below is a schema + validator with a handful of verified
worked examples (FR/ES gender nouns I can state with confidence; German order frames). The
Arabic expansion (to a real derivation/test split, replacing Exp28's 4-scene cross-DTI cells)
and any Hindi repair attempt are left as an explicit, loudly-marked TODO: I'm not fabricating
Hindi/Arabic gender-agreement word forms without a native-speaker check, per the handoff's own
flag on this (§7 item 1). The notebook will refuse to run past the gate stage until that data
is filled in.


In [1]:
import json, os

RESULTS_DIR = "exp29_results"
os.makedirs(RESULTS_DIR, exist_ok=True)

CONFIG = dict(
    # --- gates ---
    gate_acc             = 0.75,   # per-language clean-accuracy gate
    gate_min_items        = 8,
    suff_min_flip         = 0.50,  # in-language SUFFICIENCY: own-direction flip rate at alpha*, test half
    necessity_min_drop     = 0.30,  # in-language NECESSITY: knockout must drop native flip rate by >= this (absolute)
    control_max_flip       = 0.10,  # random / permutation / cue-only controls must stay <= this, cross-language
    wall_max              = 0.20,  # R <= this counts as WALL-ish
    transfer_min          = 0.50,  # R >= this counts as TRANSFER-ish
    ci_separation_required = True,  # INVERSION_SUPPORTED requires non-overlapping 95% CIs of R_gender vs R_order
    # --- DAS-via-SPSA fitting ---
    das_spsa_steps         = 300,
    das_spsa_batch         = 4,
    das_spsa_c             = 0.05,  # perturbation size, as a fraction of current direction norm
    das_spsa_lr            = 0.2,
    das_early_stop_frac     = 0.2,   # fraction of the DERIVATION half held out internally for early stopping only
    das_patience           = 30,    # SPSA steps without held-out-loss improvement before stopping
    # --- alpha* sweep (same convention as Exp27/28: smallest alpha hitting the gate) ---
    alpha_grid             = [0.5, 1, 2, 4, 8],
    # --- layer window (cheap diff-of-means pass, done BEFORE any DAS training) ---
    window_frac_of_max      = 0.5,  # layers where in-language diff-of-means effect >= this fraction of its own max
    # --- scenes / bootstrap ---
    min_scenes             = 8,
    min_xfam_scenes         = 4,
    N_BOOT                 = 2000,
    LAYER_STRIDE           = 2,
    # --- noise floor (measured empirically in self-tests; shifts below this*mult are uninterpretable) ---
    noise_floor_mult        = 1.5,
)

cfg_path = os.path.join(RESULTS_DIR, "prereg.json")
cfg_str = json.dumps(CONFIG, sort_keys=True, indent=2)
if os.path.exists(cfg_path):
    prev = open(cfg_path).read()
    if prev.strip() != cfg_str.strip():
        raise RuntimeError(
            "prereg.json already exists and differs from this run's CONFIG. "
            "A changed rule is a new study: use a new RESULTS_DIR, don't edit this one.\n"
            f"--- existing ---\n{prev}\n--- this run ---\n{cfg_str}"
        )
else:
    open(cfg_path, "w").write(cfg_str)

print("prereg frozen at", cfg_path)


prereg frozen at exp29_results/prereg.json


In [2]:
import os

if 'model' not in globals() or model is None:
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "2")  # MUST be set before importing torch
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer

    MODEL_PATH = os.environ.get("EXP29_MODEL_PATH", "/home/models/gemma-2-9b-it")
    tok = AutoTokenizer.from_pretrained(MODEL_PATH)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_PATH, torch_dtype=torch.bfloat16, attn_implementation="eager", device_map={"": 0}
    )
    model.eval()
    for p in model.parameters():
        p.requires_grad_(False)   # frozen: SPSA never needs gradients through the model
else:
    import torch  # already provided by the caller (e.g. the toy test harness)

print("model ready:", type(model).__name__)


/home/anwoy/miniconda3/envs/ke/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 464/464 [00:02<00:00, 199.40it/s]


model ready: Gemma2ForCausalLM


In [3]:
import numpy as np

def get_layers(m):
    for attr_path in [("model", "layers"), ("transformer", "h"), ("gpt_neox", "layers")]:
        obj = m
        ok = True
        for a in attr_path:
            if hasattr(obj, a):
                obj = getattr(obj, a)
            else:
                ok = False
                break
        if ok:
            return obj
    raise AttributeError("could not locate decoder layers on model")

LAYERS = get_layers(model)
N_LAYERS = len(LAYERS)

def encode_unit(prefix_text, full_text, tokenizer):
    """Tokenize full_text; verify prefix_text's tokenization is an EXACT token-prefix of it.
    Returns (full_ids, decision_pos) or None if the unit fails the prefix-consistency check
    (dropped units are counted by the caller, never silently excluded)."""
    p_ids = tokenizer(prefix_text)["input_ids"]
    f_ids = tokenizer(full_text)["input_ids"]
    if len(p_ids) > len(f_ids) or f_ids[:len(p_ids)] != p_ids:
        return None
    return f_ids, len(p_ids) - 1

def collate(id_lists, pad_id):
    T = max(len(x) for x in id_lists)
    ids = np.full((len(id_lists), T), pad_id, dtype=np.int64)
    mask = np.zeros((len(id_lists), T), dtype=np.int64)
    for i, x in enumerate(id_lists):
        ids[i, :len(x)] = x
        mask[i, :len(x)] = 1
    device = next(model.parameters()).device if ('model' in globals() and model is not None and hasattr(model, "parameters")) else torch.device("cpu")
    return torch.tensor(ids, device=device), torch.tensor(mask, device=device)

HOOK_REGISTRY = []

def _clear_hooks():
    for h in HOOK_REGISTRY:
        h.remove()
    HOOK_REGISTRY.clear()

def make_intervention_hook(mode, pos_list, vec=None, alpha=0.0, sigma=1.0):
    """mode: 'add'      h[i] += alpha*sigma*unit(vec)   (sufficiency / steering)
              'knockout' h[i] -= (h[i].u)u                (necessity / ablation)
       pos_list: per-row decision position (so a batch can mix scene lengths)."""
    def fn(module, inp, out):
        is_tuple = isinstance(out, tuple)
        h = out[0] if is_tuple else out
        hc = h.clone()
        u = None
        if vec is not None:
            nv = np.linalg.norm(vec) + 1e-8
            u_f = torch.tensor(vec / nv, dtype=torch.float32, device=hc.device); u = u_f.to(hc.dtype)
        for i, p in enumerate(pos_list):
            row = hc[i, p, :]
            if mode == "add":
                row = row + float(alpha) * float(sigma) * u
            elif mode == "knockout":
                row = (row.float() - (row.float() @ u_f) * u_f).to(hc.dtype)
            else:
                raise ValueError(mode)
            hc[i, p, :] = row
        return (hc,) + out[1:] if is_tuple else hc
    return fn

def forward_with_intervention(ids, mask, layer_idx, pos_list, mode, vec=None, alpha=0.0, sigma=1.0):
    device = next(model.parameters()).device if hasattr(model, "parameters") else ids.device
    ids, mask = ids.to(device), mask.to(device)
    _clear_hooks()
    h = LAYERS[layer_idx].register_forward_hook(
        make_intervention_hook(mode, pos_list, vec=vec, alpha=alpha, sigma=sigma)
    )
    HOOK_REGISTRY.append(h)
    try:
        with torch.no_grad():
            out = model(input_ids=ids, attention_mask=mask, use_cache=False)
        return out.logits
    finally:
        _clear_hooks()

def clean_forward(ids, mask):
    device = next(model.parameters()).device if hasattr(model, "parameters") else ids.device
    ids, mask = ids.to(device), mask.to(device)
    _clear_hooks()
    with torch.no_grad():
        return model(input_ids=ids, attention_mask=mask, use_cache=False).logits

def capture_states(ids, mask, pos_list):
    device = next(model.parameters()).device if hasattr(model, "parameters") else ids.device
    ids, mask = ids.to(device), mask.to(device)
    """Hooks every layer; returns states[N, n_layers, d] at each row's own decision position."""
    captured = [None] * N_LAYERS
    def mk(li):
        def fn(mod, inp, out):
            captured[li] = (out[0] if isinstance(out, tuple) else out).detach().float().cpu()
        return fn
    _clear_hooks()
    for li, L in enumerate(LAYERS):
        HOOK_REGISTRY.append(L.register_forward_hook(mk(li)))
    try:
        with torch.no_grad():
            model(input_ids=ids, attention_mask=mask, use_cache=False)
    finally:
        _clear_hooks()
    N = ids.shape[0]
    d = captured[0].shape[-1]
    out = np.zeros((N, N_LAYERS, d))
    for li in range(N_LAYERS):
        h = captured[li].numpy()
        for i in range(N):
            out[i, li, :] = h[i, pos_list[i], :]
    return out

def residual_norm(states_2d):
    # states_2d: [N, d] -> mean L2 norm (sigma_L for a given layer)
    return float(np.sqrt((states_2d ** 2).sum(-1)).mean())

def score_candidates(logits, cand_id_lists, pos_list):
    """Teacher-forced sum log-prob of each row's candidate continuation, starting at pos+1."""
    lg = logits.detach().float().cpu().numpy() if hasattr(logits, "numpy") else logits
    out = np.zeros(len(cand_id_lists))
    for i, cand in enumerate(cand_id_lists):
        p = pos_list[i]
        row = lg[i] - lg[i].max(-1, keepdims=True)
        logp = row - np.log(np.exp(row).sum(-1, keepdims=True))
        s, cur = 0.0, p
        for tid in cand:
            s += logp[cur, tid]
            cur += 1
        out[i] = s
    return out

def margin(logits, posA_ids, posB_ids, pos_list):
    """S = logP(class A continuation) - logP(class B continuation), per row."""
    return score_candidates(logits, posA_ids, pos_list) - score_candidates(logits, posB_ids, pos_list)

def hidden_dim():
    """Robust hidden-size lookup: real HF config first, toy-harness shim shape as fallback."""
    if hasattr(model, "config") and hasattr(model.config, "hidden_size"):
        return model.config.hidden_size
    return LAYERS[0].W1.shape[0]  # toy-harness Layer exposes W1: (D, 2D)

print(f"infra ready: {N_LAYERS} layers, hidden_dim={hidden_dim()}")


infra ready: 42 layers, hidden_dim=3584


In [4]:
# ---------------------------------------------------------------------------
# GENDER item bank: schema + validator + a full 20-concept x 4-language bank.
#
# CONFIDENCE LEVELS (read before trusting any number that comes out of this
# notebook -- this is the single most important caveat in the whole file):
#   FR / ES  -- standard, high-confidence dictionary gender for common nouns.
#               Low risk.
#   HI / AR  -- NOT reviewed by a native speaker. Two tiers:
#     (a) web-cross-checked this session against real grammar references
#         (table, house, window, car, sun, book, tooth) -- one of these
#         (Arabic "tooth") was WRONG on first pass (I had it feminine; it's
#         masculine, confirmed via arabic.fi and madinaharabic) and is fixed
#         below. That correction is itself evidence this tier still needs a
#         human pass, not proof the rest is safe.
#     (b) built from well-documented morphological patterns (Arabic: ends in
#         ta marbuta -> feminine, else masculine-by-default, checked against
#         the known short list of irregular feminine-without-ta-marbuta
#         exceptions; Hindi: -aa/-ii ending heuristic plus common-word
#         recall) but NOT individually cross-checked against a source this
#         session.
#   Every HI/AR row below is tagged verified=False regardless of tier --
#   that flag means "native-speaker signed off", which hasn't happened.
#   Get a native-speaker pass on at least tier (b) before trusting a
#   published claim built on AR/HI cells. The clean-accuracy gate below is
#   an empirical check on the MODEL's behavior, not a check on whether
#   these translations are actually correct -- a wrong-but-systematic
#   translation could still pass the gate.
# ---------------------------------------------------------------------------
# Unit schema (one row per concept x language):
#   concept   : str, concept id (same id across languages -- not required to
#               share gender; used only for the xfam flag and bookkeeping)
#   lang      : 'FR' | 'ES' | 'HI' | 'AR'
#   prefix    : str, text up to (not including) the agreeing adjective
#   fem       : str, feminine adjective form + fixed terminator
#   masc      : str, masculine adjective form + fixed terminator
#   native    : 'fem' | 'masc' -- this concept's actual gender in this language
#   xfam      : bool -- True if this concept's gender DIFFERS between FR and ES
#   split     : 'derivation' | 'test'
#   verified  : bool -- native-speaker sign-off. True for FR/ES. False for
#               every HI/AR row (see confidence note above).

GENDER_TERM = {"FR": ".", "ES": ".", "HI": " है.", "AR": "."}
GENDER_ADJ = {
    "FR": {"fem": "belle", "masc": "beau"},
    "ES": {"fem": "bella", "masc": "bello"},
    "HI": {"fem": "अच्छी", "masc": "अच्छा"},
    "AR": {"fem": "كبيرة", "masc": "كبير"},
}

# (concept, split, {lang: (prefix, native_gender, verified)})
# Hindi prefix pattern: "यह <noun> बहुत" ("this <noun> very") + adjective + " है."
# Arabic prefix pattern: definite noun ("al-" + noun) + adjective + "."
_GENDER_TABLE = [
    ("river",    "derivation", {"FR": ("La riviere est", "fem", True),
                                 "ES": ("El rio es", "masc", True),
                                 "HI": ("यह नदी बहुत", "fem", False),
                                 "AR": ("النهر", "masc", False)}),
    ("wall",     "derivation", {"FR": ("Le mur est", "masc", True),
                                 "ES": ("La pared es", "fem", True),
                                 "HI": ("यह दीवार बहुत", "fem", False),
                                 "AR": ("الجدار", "masc", False)}),
    ("bag",      "derivation", {"FR": ("Le sac est", "masc", True),
                                 "ES": ("La bolsa es", "fem", True),
                                 "HI": ("यह झोला बहुत", "masc", False),
                                 "AR": ("الحقيبة", "fem", False)}),
    ("plate",    "derivation", {"FR": ("L'assiette est", "fem", True),
                                 "ES": ("El plato es", "masc", True),
                                 "HI": ("यह थाली बहुत", "fem", False),
                                 "AR": ("الصحن", "masc", False)}),
    ("book",     "derivation", {"FR": ("Le livre est", "masc", True),
                                 "ES": ("El libro es", "masc", True),
                                 "HI": ("यह किताब बहुत", "fem", False),   # web-checked: kitab is the standard fem word for book
                                 "AR": ("الكتاب", "masc", False)}),       # web-checked: kitab = masculine (arabic.fi, madinaharabic)
    ("table",    "derivation", {"FR": ("La table est", "fem", True),
                                 "ES": ("La mesa es", "fem", True),
                                 "HI": ("यह मेज़ बहुत", "fem", False),     # web-checked: mez = unmarked feminine (hindipod101)
                                 "AR": ("الطاولة", "fem", False)}),
    ("door",     "derivation", {"FR": ("La porte est", "fem", True),
                                 "ES": ("La puerta es", "fem", True),
                                 "HI": ("यह दरवाज़ा बहुत", "masc", False),
                                 "AR": ("الباب", "masc", False)}),
    ("window",   "derivation", {"FR": ("La fenetre est", "fem", True),
                                 "ES": ("La ventana es", "fem", True),
                                 "HI": ("यह खिड़की बहुत", "fem", False),   # web-checked: khirki = feminine (hindipod101)
                                 "AR": ("النافذة", "fem", False)}),
    ("house",    "derivation", {"FR": ("La maison est", "fem", True),
                                 "ES": ("La casa es", "fem", True),
                                 "HI": ("यह घर बहुत", "masc", False),      # web-checked: ghar = unmarked masculine (hindipod101)
                                 "AR": ("البيت", "masc", False)}),        # web-checked: bayt = masculine (note: "daar" would be an irregular-fem exception; bayt avoids that trap)
    ("sun",      "derivation", {"FR": ("Le soleil est", "masc", True),
                                 "ES": ("El sol es", "masc", True),
                                 "HI": ("यह सूरज बहुत", "masc", False),    # web-checked: suraj = masculine (lingq)
                                 "AR": ("الشمس", "fem", False)}),         # web-checked: shams = irregular feminine (arabic-for-nerds, kalimah-center, multiple sources)
    ("ring",     "test",       {"FR": ("La bague est", "fem", True),
                                 "ES": ("El anillo es", "masc", True),
                                 "HI": ("यह अंगूठी बहुत", "fem", False),
                                 "AR": ("الخاتم", "masc", False)}),
    ("tooth",    "test",       {"FR": ("La dent est", "fem", True),
                                 "ES": ("El diente es", "masc", True),
                                 "HI": ("यह दांत बहुत", "masc", False),
                                 "AR": ("السن", "masc", False)}),         # CORRECTED: web-checked masculine (arabic.fi, madinaharabic) -- first draft had this wrong as feminine
    ("car",      "test",       {"FR": ("La voiture est", "fem", True),
                                 "ES": ("El coche es", "masc", True),
                                 "HI": ("यह गाड़ी बहुत", "fem", False),    # web-checked: gaadi = feminine (lingq)
                                 "AR": ("السيارة", "fem", False)}),
    ("forest",   "test",       {"FR": ("La foret est", "fem", True),
                                 "ES": ("El bosque es", "masc", True),
                                 "HI": ("यह जंगल बहुत", "masc", False),
                                 "AR": ("الغابة", "fem", False)}),
    ("room",     "test",       {"FR": ("La chambre est", "fem", True),
                                 "ES": ("La habitacion es", "fem", True),
                                 "HI": ("यह कमरा बहुत", "masc", False),
                                 "AR": ("الغرفة", "fem", False)}),
    ("road",     "test",       {"FR": ("La rue est", "fem", True),
                                 "ES": ("La calle es", "fem", True),
                                 "HI": ("यह सड़क बहुत", "fem", False),
                                 "AR": ("الشارع", "masc", False)}),
    ("moon",     "test",       {"FR": ("La lune est", "fem", True),
                                 "ES": ("La luna es", "fem", True),
                                 "HI": ("यह चांद बहुत", "masc", False),
                                 "AR": ("القمر", "masc", False)}),        # web-checked: qamar = masculine (englishtoarabic.org, explicitly contrasted with shams)
    ("bridge",   "test",       {"FR": ("Le pont est", "masc", True),
                                 "ES": ("El puente es", "masc", True),
                                 "HI": ("यह पुल बहुत", "masc", False),
                                 "AR": ("الجسر", "masc", False)}),
    ("lake",     "test",       {"FR": ("Le lac est", "masc", True),
                                 "ES": ("El lago es", "masc", True),
                                 "HI": ("यह झील बहुत", "fem", False),
                                 "AR": ("البحيرة", "fem", False)}),
    ("mountain", "test",       {"FR": ("La montagne est", "fem", True),
                                 "ES": ("La montana es", "fem", True),
                                 "HI": ("यह पहाड़ बहुत", "masc", False),
                                 "AR": ("الجبل", "masc", False)}),
]

def _xfam(entry):
    fr = entry["FR"][1]; es = entry["ES"][1]
    return fr != es

GENDER_UNITS = []
for concept, split, entry in _GENDER_TABLE:
    xfam = _xfam(entry)
    for lang, (prefix, native, verified) in entry.items():
        adj = GENDER_ADJ[lang]
        GENDER_UNITS.append(dict(
            concept=concept, lang=lang, prefix=prefix,
            fem=adj["fem"] + GENDER_TERM[lang], masc=adj["masc"] + GENDER_TERM[lang],
            native=native, xfam=xfam, split=split, verified=verified,
        ))

REQUIRED_PER_LANG_PER_SPLIT = 8   # derivation and test each, per language
MIN_PER_CLASS_PER_SPLIT = 2       # both fem and masc must appear at least this often

def validate_gender_bank(units, langs=("FR", "ES", "HI", "AR")):
    by = {}
    for u in units:
        by.setdefault((u["lang"], u["split"]), []).append(u)
    missing = []
    for lang in langs:
        for split in ("derivation", "test"):
            rows = by.get((lang, split), [])
            n = len(rows)
            if n < REQUIRED_PER_LANG_PER_SPLIT:
                missing.append(f"{lang}/{split}: have {n}, need >= {REQUIRED_PER_LANG_PER_SPLIT}")
                continue
            n_fem = sum(1 for r in rows if r["native"] == "fem")
            n_masc = n - n_fem
            if n_fem < MIN_PER_CLASS_PER_SPLIT or n_masc < MIN_PER_CLASS_PER_SPLIT:
                missing.append(f"{lang}/{split}: class imbalance (fem={n_fem}, masc={n_masc}), "
                                f"need >= {MIN_PER_CLASS_PER_SPLIT} of each")
    n_xfam_deriv = sum(1 for u in units if u["xfam"] and u["split"] == "derivation" and u["lang"] == "FR")
    n_xfam_test  = sum(1 for u in units if u["xfam"] and u["split"] == "test" and u["lang"] == "FR")
    if n_xfam_deriv < CONFIG["min_xfam_scenes"]:
        missing.append(f"xfam/derivation: have {n_xfam_deriv}, need >= {CONFIG['min_xfam_scenes']}")
    if n_xfam_test < CONFIG["min_xfam_scenes"]:
        missing.append(f"xfam/test: have {n_xfam_test}, need >= {CONFIG['min_xfam_scenes']}")
    return missing

_missing = validate_gender_bank(GENDER_UNITS)
GENDER_BANK_READY = (len(_missing) == 0)
n_unverified = sum(1 for u in GENDER_UNITS if not u["verified"])
if not GENDER_BANK_READY:
    print("GENDER item bank FAILS the count/balance check:")
    for m in _missing:
        print("  -", m)
else:
    print(f"GENDER item bank passes the count/balance check: {len(GENDER_UNITS)} rows, "
          f"{n_unverified} not natively verified (all HI/AR rows -- see confidence note above).")
    print("The clean-accuracy gate (Cell 6+) is the next real check; a native-speaker pass")
    print("on the HI/AR rows is still owed before trusting a published claim.")


GENDER item bank passes the count/balance check: 80 rows, 40 not natively verified (all HI/AR rows -- see confidence note above).
The clean-accuracy gate (Cell 6+) is the next real check; a native-speaker pass
on the HI/AR rows is still owed before trusting a published claim.


In [5]:
# ---------------------------------------------------------------------------
# ORDER item bank: German in-language ceiling frames (V2 vs subordinate-clause)
# PLUS the new nuisance 'ob' frames (Section 5.3.3 of the handoff): same
# verb-final shape as 'dass', but no order change, so a real order direction
# must beat it as a control.
# ---------------------------------------------------------------------------
# DE_FRAMES schema: scene (int), kind in {'V2a','V2b','SUBa','SUBb','SUBc','SUBd'},
#   prefix (up to the decision token), hl ('head-last' continuation ids source),
#   hf ('head-first' continuation ids source). V2a/V2b = head-first (verb-second
#   main clause); SUBa/SUBb = head-last ('dass'-clause); SUBc/SUBd = head-last
#   nuisance ('ob'-clause / glauben-denken lexical variants) -- same surface
#   shape as SUBa/SUBb, no in-language order ceiling expected.

DE_FRAMES = [
    dict(scene=0, kind="V2a",  prefix="Der Mann sieht den Hund",   order="head_first"),
    dict(scene=0, kind="SUBa", prefix="Ich glaube, dass der Mann den Hund sieht", order="head_last"),
    dict(scene=0, kind="SUBc", prefix="Ich weiss nicht, ob der Mann den Hund sieht", order="head_last_nuisance"),
    dict(scene=1, kind="V2b",  prefix="Die Frau liest das Buch",   order="head_first"),
    dict(scene=1, kind="SUBb", prefix="Ich glaube, dass die Frau das Buch liest", order="head_last"),
    dict(scene=1, kind="SUBd", prefix="Ich weiss nicht, ob die Frau das Buch liest", order="head_last_nuisance"),
]

# CLAUSE bank (cross-language recipients): verb-object vs object-verb candidates,
# subject-prefix. TODO -- only EN/DE worked here with confidence; FR/ES/HI/JA/KO/TR/ZH
# need the original Exp27 CLAUSE bank re-attached (it is not reproduced verbatim here --
# this notebook is deliberately fresh/self-contained per the handoff's own working-style
# rule, so re-derive or re-paste it from Exp27_A_WordOrder_Gate.ipynb, item-by-item
# checked, rather than hand-retyping from memory).
CLAUSE_BANK = [
    dict(lang="EN", scene=0, prefix="The man", cand_vo="sees the dog.", cand_ov="the dog sees.", native="head_first"),
    dict(lang="DE", scene=0, prefix="Der Mann", cand_vo="sieht den Hund.", cand_ov="den Hund sieht.", native="head_first"),
]

REQUIRED_SCENES = 8

def validate_order_bank(de_frames, clause_bank, langs_expected=("EN","FR","ES","DE","HI","JA","KO","TR")):
    missing = []
    n_scenes = len({f["scene"] for f in de_frames})
    if n_scenes < REQUIRED_SCENES:
        missing.append(f"DE_FRAMES: {n_scenes} scenes, need >= {REQUIRED_SCENES}")
    have_langs = {u["lang"] for u in clause_bank}
    for lang in langs_expected:
        if lang not in have_langs:
            missing.append(f"CLAUSE_BANK: no items for {lang}")
    return missing

_missing_order = validate_order_bank(DE_FRAMES, CLAUSE_BANK)
ORDER_BANK_READY = (len(_missing_order) == 0)
if not ORDER_BANK_READY:
    print("ORDER item bank is a SCAFFOLD ONLY -- not enough items to run yet.")
    for m in _missing_order:
        print("  -", m)
    print()
    print("Re-attach the real Exp27 CLAUSE_BANK (12 scenes x 8 languages) and expand")
    print("DE_FRAMES to >= 8 scenes before running past the gate stage.")
else:
    print("ORDER item bank passes the count check.")


ORDER item bank is a SCAFFOLD ONLY -- not enough items to run yet.
  - DE_FRAMES: 2 scenes, need >= 8
  - CLAUSE_BANK: no items for FR
  - CLAUSE_BANK: no items for ES
  - CLAUSE_BANK: no items for HI
  - CLAUSE_BANK: no items for JA
  - CLAUSE_BANK: no items for KO
  - CLAUSE_BANK: no items for TR

Re-attach the real Exp27 CLAUSE_BANK (12 scenes x 8 languages) and expand
DE_FRAMES to >= 8 scenes before running past the gate stage.


In [6]:
def derivation_test_split(units, key_fn, n_derivation=None):
    """units already carry a 'split' field for gender (balanced by xfam at authoring time);
    for order, split by scene index: first half -> derivation, second half -> test."""
    deriv = [u for u in units if u.get("split") == "derivation"]
    test  = [u for u in units if u.get("split") == "test"]
    return deriv, test

def order_split_by_scene(frames, cfg):
    scenes = sorted({f["scene"] for f in frames})
    mid = len(scenes) // 2
    deriv_scenes, test_scenes = set(scenes[:mid]), set(scenes[mid:])
    deriv = [f for f in frames if f["scene"] in deriv_scenes]
    test  = [f for f in frames if f["scene"] in test_scenes]
    return deriv, test

def clean_accuracy_gate(units_for_lang, tokenizer, cfg):
    """Fraction of items where the native class scores higher than the opposite class,
    clean (unpatched) run. Returns (accuracy, n_usable, n_dropped_tokenization)."""
    if not units_for_lang:
        return 0.0, 0, 0
    kept, dropped = [], 0
    for u in units_for_lang:
        enc_f = encode_unit(u["prefix"], u["prefix"] + " " + u["fem"], tokenizer) if "fem" in u else None
        kept.append(u)  # placeholder: real notebook wires this through score_candidates with margin()
    n = len(kept)
    # NOTE: full scoring wired in Cell 15 (run pipeline) once a real tokenizer/model are present;
    # this function's signature and gate semantics are frozen here so the gate rule itself is
    # part of the pre-registration, independent of which model it's later run against.
    return None, n, dropped

print("gate + split functions ready (full scoring wired against a real tokenizer in the run cell)")


gate + split functions ready (full scoring wired against a real tokenizer in the run cell)


In [7]:
def diff_of_means_direction(states_class_a, states_class_b):
    """states_class_a/b: [N, d] at one layer. Returns unnormalized mean-difference vector."""
    return states_class_a.mean(0) - states_class_b.mean(0)

def in_language_curve(states_all_layers_a, states_all_layers_b, margin_fn):
    """states_*: [N, n_layers, d]. Returns per-layer in-language effect using margin_fn,
    which scores how well each layer's diff-of-means direction separates the two classes
    on a held-out slice (caller passes derivation-internal held-out states)."""
    n_layers = states_all_layers_a.shape[1]
    effect = np.zeros(n_layers)
    for li in range(n_layers):
        d = diff_of_means_direction(states_all_layers_a[:, li, :], states_all_layers_b[:, li, :])
        effect[li] = margin_fn(d, li)
    return effect

def select_window(effect_curve, cfg):
    """Layers where effect >= window_frac_of_max * max(effect). Returns sorted layer indices.
    This selection happens on the DERIVATION half only -- never on cross-language results --
    per the frozen rule in the handoff (Section 5.1)."""
    if effect_curve.max() <= 0:
        return []
    thresh = cfg["window_frac_of_max"] * effect_curve.max()
    return sorted(int(i) for i in np.where(effect_curve >= thresh)[0])

print("diff-of-means window-selection ready")


diff-of-means window-selection ready


In [8]:
def spsa_fit_direction(init_vec, loss_fn, cfg, holdout_loss_fn=None, rng=None):
    """Zeroth-order (SPSA) fit of a unit direction that minimizes loss_fn(direction).
    Needs only forward passes (via loss_fn), no autograd -- same code path works against
    the real model and the toy-harness shim.

    loss_fn(vec) -> scalar (lower is better), evaluated on the DERIVATION-internal
        training slice.
    holdout_loss_fn(vec) -> scalar, evaluated on the DERIVATION-internal held-out slice,
        used only for early stopping (never touches the test half).
    Returns (direction, history) where direction is unit-norm.
    """
    rng = rng or np.random.default_rng(0)
    d = init_vec.shape[0]
    theta = init_vec / (np.linalg.norm(init_vec) + 1e-8)
    best_holdout, best_theta, patience_left = np.inf, theta.copy(), cfg["das_patience"]
    history = []
    for step in range(cfg["das_spsa_steps"]):
        delta = rng.choice([-1.0, 1.0], size=d)
        c = cfg["das_spsa_c"]
        loss_plus  = loss_fn(theta + c * delta)
        loss_minus = loss_fn(theta - c * delta)
        ghat = (loss_plus - loss_minus) / (2.0 * c) * delta  # unbiased gradient estimate
        theta = theta - cfg["das_spsa_lr"] * ghat
        theta = theta / (np.linalg.norm(theta) + 1e-8)
        history.append(float((loss_plus + loss_minus) / 2.0))
        if holdout_loss_fn is not None:
            hl = holdout_loss_fn(theta)
            if hl < best_holdout - 1e-6:
                best_holdout, best_theta, patience_left = hl, theta.copy(), cfg["das_patience"]
            else:
                patience_left -= 1
                if patience_left <= 0:
                    break
    final = best_theta if holdout_loss_fn is not None else theta
    return final / (np.linalg.norm(final) + 1e-8), history

print("SPSA direction-fitting ready (dimension-independent: 2 forward passes / step)")


SPSA direction-fitting ready (dimension-independent: 2 forward passes / step)


In [9]:
def calibrate_alpha_star(direction, layer_idx, deriv_units, sigma, cfg, flip_eval_fn):
    """Smallest alpha in cfg['alpha_grid'] at which the SOURCE language's own in-language
    flip rate (derivation half) reaches >= cfg['suff_min_flip']. flip_eval_fn(direction,
    layer_idx, alpha, sigma) -> flip_rate in [0,1]."""
    for alpha in cfg["alpha_grid"]:
        fr = flip_eval_fn(direction, layer_idx, alpha, sigma)
        if fr >= cfg["suff_min_flip"]:
            return alpha, fr
    return cfg["alpha_grid"][-1], flip_eval_fn(direction, layer_idx, cfg["alpha_grid"][-1], sigma)

print("alpha* calibration ready")


alpha* calibration ready


In [10]:
def sufficiency_effect(direction, layer_idx, alpha, sigma, units, tokenizer, metric="flip"):
    """ADD direction*alpha*sigma at layer_idx/decision-token; measure shift_toward / flip
    relative to clean run, on the given units (already encoded + collated by caller)."""
    raise NotImplementedError("wired against real tokenizer/model + encoded unit batches in the run cell")

def necessity_effect(direction, layer_idx, units, tokenizer, metric="flip"):
    """KNOCKOUT (project out) direction at layer_idx/decision-token on the SOURCE language's
    OWN units; measure how much the native-class preference degrades vs clean. A real,
    causally-load-bearing direction should show necessity_min_drop or more."""
    raise NotImplementedError("wired against real tokenizer/model + encoded unit batches in the run cell")

print("sufficiency/necessity scaffolds ready (NotImplementedError until wired to real units in Cell 15)")


sufficiency/necessity scaffolds ready (NotImplementedError until wired to real units in Cell 15)


In [11]:
def random_directions(d, n, rng=None):
    rng = rng or np.random.default_rng(1)
    V = rng.normal(size=(n, d))
    V /= np.linalg.norm(V, axis=1, keepdims=True) + 1e-8
    return V

def permutation_directions(states_a, states_b, n, rng=None):
    """Re-derive diff-of-means directions after shuffling the class labels across the
    pooled (a+b) items, n times -- null distribution for 'direction is just item/concept
    noise', per Section 5.3.2."""
    rng = rng or np.random.default_rng(2)
    pooled = np.concatenate([states_a, states_b], axis=0)
    n_a = states_a.shape[0]
    out = []
    for _ in range(n):
        idx = rng.permutation(pooled.shape[0])
        a_idx, b_idx = idx[:n_a], idx[n_a:]
        out.append(pooled[a_idx].mean(0) - pooled[b_idx].mean(0))
    return np.stack(out)

def cue_only_direction(states_lang_x_neutral, states_lang_y_neutral):
    """A REAL language-identity direction: mean state on a NEUTRAL task (unrelated to
    gender/order) in language X minus language Y. Used as a cue-only falsifier per the
    triangulation standard -- stronger than norm-matched random noise, because it
    specifically carries 'this came from a different language' rather than generic noise.
    If patching THIS moves the gender/order readout about as much as the real direction
    does, the apparent transfer may just be 'patching anything language-flavored nudges
    this readout', which random-direction controls would not catch."""
    return states_lang_x_neutral.mean(0) - states_lang_y_neutral.mean(0)

def specificity_2x2(effect_own_task, effect_cross_task, min_ratio=3.0):
    """A shared code should show own-task effect >= min_ratio * cross-task effect
    (Section 5.3.4): applying d_g to the ORDER readout and d_o to the GENDER readout."""
    if effect_cross_task <= 1e-8:
        return effect_own_task > 0, float('inf')
    ratio = effect_own_task / effect_cross_task
    return ratio >= min_ratio, ratio

print("control functions ready: random, permutation, nuisance-direction (reuse sufficiency_effect),"
      " 2x2 specificity, and a real cue-only language-ID falsifier")


control functions ready: random, permutation, nuisance-direction (reuse sufficiency_effect), 2x2 specificity, and a real cue-only language-ID falsifier


In [12]:
def bootstrap_ci(values_by_scene, stat_fn, n_boot, rng=None, alpha=0.05):
    """Resample SCENES jointly (not items), per Section 5.2's convention.
    values_by_scene: list of per-scene value arrays (already grouped by scene)."""
    rng = rng or np.random.default_rng(3)
    n = len(values_by_scene)
    boots = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        sample = [values_by_scene[i] for i in idx]
        boots.append(stat_fn(sample))
    boots = np.sort(np.array(boots))
    lo = boots[int((alpha / 2) * n_boot)]
    hi = boots[int((1 - alpha / 2) * n_boot) - 1]
    return float(np.mean(boots)), float(lo), float(hi)

def transfer_ratio(effect_recipient, effect_source_inlang):
    if effect_source_inlang <= 1e-8:
        return None  # source gate should have already failed upstream of this
    return effect_recipient / effect_source_inlang

print("bootstrap + transfer-ratio functions ready (scene-level resampling)")


bootstrap + transfer-ratio functions ready (scene-level resampling)


In [13]:
def classify_cell(suff_ok, necessity_ok, controls_ok, R, R_ci, cfg):
    """A cell can only be called TRANSFER if BOTH sufficiency and necessity gates pass
    AND controls stayed within bound. This is the change from the original Exp29 proposal
    (sufficiency-only) to the triangulation-gated version."""
    if not (suff_ok and necessity_ok and controls_ok):
        return "UNUSABLE"
    if R is None:
        return "NO_CEILING"
    lo, hi = R_ci
    if R >= cfg["transfer_min"]:
        return "TRANSFER"
    if R <= cfg["wall_max"]:
        return "WALL"
    return "AMBIGUOUS"

def verdict(cell_report, cfg):
    """cell_report: list of dicts with at least {'task': 'gender'|'order', 'recipient':...,
    'class': one of TRANSFER/WALL/AMBIGUOUS/NO_CEILING/UNUSABLE, 'R':..., 'R_ci': (lo,hi)}.
    Mirrors the Section 5.4 decision tree, now requiring the necessity+sufficiency gates
    baked into classify_cell (above) to have already passed for every cell counted here."""
    gender = [c for c in cell_report if c["task"] == "gender" and c["recipient"] in ("FR", "ES")]
    order  = [c for c in cell_report if c["task"] == "order"  and c["recipient"] in ("FR", "ES")]

    if any(c["class"] == "UNUSABLE" for c in gender + order):
        return "INSTRUMENT_INADEQUATE: at least one required FR/ES cell is UNUSABLE (gate, sufficiency, necessity, or control failure)."
    if len(gender) < 2 or len(order) < 2:
        return "INCONCLUSIVE: fewer than 2 usable FR/ES cells for gender and/or order."

    def ok_transfer(cells):
        return all(c["class"] == "TRANSFER" for c in cells)
    def ok_wall(cells):
        return all(c["class"] == "WALL" for c in cells)
    def cis_separate(cells_a, cells_b):
        a_lo = min(c["R_ci"][0] for c in cells_a); a_hi = max(c["R_ci"][1] for c in cells_a)
        b_lo = min(c["R_ci"][0] for c in cells_b); b_hi = max(c["R_ci"][1] for c in cells_b)
        return (a_lo > b_hi) or (b_lo > a_hi)

    order_ge_gender = all(
        o["R"] is not None and g["R"] is not None and o["R"] >= g["R"]
        for o, g in zip(sorted(order, key=lambda c: c["recipient"]), sorted(gender, key=lambda c: c["recipient"]))
    )

    # Priority matters here and is itself part of the frozen rule (not an incidental
    # implementation detail): INVERSION_SUPPORTED and BOTH_SHARED are checked first because
    # they're the most specific/determinate findings. REVERSED is checked before BOTH_LOW
    # because BOTH_LOW only inspects gender's own R -- without this ordering, a clean
    # order-beats-gender result (gender WALL, order TRANSFER, CIs separate) would be caught
    # by BOTH_LOW's "gender R <= wall_max" clause before REVERSED ever got evaluated, which
    # would silently discard the more informative REVERSED finding. Caught by the toy-harness
    # branch-forcing self-test below -- do not reorder these without re-running that test.
    if ok_transfer(gender) and ok_wall(order) and (not cfg["ci_separation_required"] or cis_separate(gender, order)):
        return "INVERSION_SUPPORTED: gender transfers, order does not, CIs separate, controls and 2x2 specificity passed."
    if ok_transfer(gender) and ok_transfer(order):
        return "BOTH_SHARED: both gender and order transfer -- not the inversion."
    if order_ge_gender and cis_separate(order, gender):
        return "REVERSED: order transfers as much as or more than gender, CIs separate."
    if (not ok_transfer(gender)) and all(c["R"] is not None and c["R"] <= cfg["wall_max"] for c in gender):
        return "BOTH_LOW: gender does not transfer either -- the contrast fails."
    return "INCONCLUSIVE: no branch condition met -- report cell-level results, no post-hoc rule changes."

print("verdict logic ready")


verdict logic ready


In [14]:
# (1) additive no-op at alpha=0
# (2) knockout-of-zero-direction no-op
# (3) hooks are removed after every call (registry empty post-call)
# (4) SPSA loss should (on average) decrease on a toy convex-ish target
# These run against a TINY synthetic batch built straight from the tokenizer -- independent
# of the gender/order item banks, so they validate the intervention mechanics on their own,
# before any real GENDER_UNITS/DE_FRAMES data is filled in.

def _self_test_batch(n=3):
    texts = ["hello world test", "a second example", "and one more"][:n]
    ids_list = [tok(t)["input_ids"] for t in texts]
    pad_id = getattr(tok, "pad_token_id", 0) or 0
    ids, mask = collate(ids_list, pad_id)
    pos_list = [len(x) - 1 for x in ids_list]
    return ids, mask, pos_list

def self_test_hooks_cleared():
    assert len(HOOK_REGISTRY) == 0, "hook registry not empty after a call -- hooks are leaking"

def self_test_add_noop():
    ids, mask, pos_list = _self_test_batch()
    d = hidden_dim()
    base = clean_forward(ids, mask)
    patched = forward_with_intervention(ids, mask, 0, pos_list, "add", vec=np.ones(d), alpha=0.0, sigma=1.0)
    bn = base.detach().float().cpu().numpy() if hasattr(base, "numpy") else base
    pn = patched.detach().float().cpu().numpy() if hasattr(patched, "numpy") else patched
    delta = float(np.abs(bn - pn).max())
    assert delta < 1e-4, f"alpha=0 add was not a no-op, max|delta|={delta}"
    self_test_hooks_cleared()
    return delta

def self_test_knockout_zero_noop():
    ids, mask, pos_list = _self_test_batch()
    d = hidden_dim()
    base = clean_forward(ids, mask)
    patched = forward_with_intervention(ids, mask, 0, pos_list, "knockout", vec=np.zeros(d))
    bn = base.detach().float().cpu().numpy() if hasattr(base, "numpy") else base
    pn = patched.detach().float().cpu().numpy() if hasattr(patched, "numpy") else patched
    delta = float(np.abs(bn - pn).max())
    assert delta < 1e-4, f"knockout of a zero direction was not a no-op, max|delta|={delta}"
    self_test_hooks_cleared()
    return delta

def self_test_knockout_orthogonalizes():
    """After knockout along u, the component of the patched state along u should be ~0."""
    ids, mask, pos_list = _self_test_batch()
    d = hidden_dim()
    rng = np.random.default_rng(7)
    u = rng.normal(size=d); u /= np.linalg.norm(u)
    states_before = capture_states(ids, mask, pos_list)[:, 0, :]  # layer 0, [N, d]
    forward_with_intervention(ids, mask, 0, pos_list, "knockout", vec=u)
    # re-capture AFTER the (already-cleared) hook ran once more, this time keeping it live:
    _clear_hooks()
    hook = LAYERS[0].register_forward_hook(make_intervention_hook("knockout", pos_list, vec=u))
    HOOK_REGISTRY.append(hook)
    try:
        captured = {}
        def grab(mod, inp, out):
            captured["h"] = (out[0] if isinstance(out, tuple) else out).detach().float().cpu()
        h2 = LAYERS[0].register_forward_hook(grab)
        HOOK_REGISTRY.append(h2)
        with torch.no_grad():
            model(input_ids=ids, attention_mask=mask, use_cache=False)
    finally:
        _clear_hooks()
    after = captured["h"].numpy() if hasattr(captured["h"], "numpy") else captured["h"]
    resid = max(abs(float(after[i, pos_list[i], :] @ u)) for i in range(len(pos_list)))
    assert resid < 0.05, f"knockout left a residual component along u of {resid}"
    return resid

def self_test_spsa_decreases(d=8, steps=60, seed=0):
    rng = np.random.default_rng(seed)
    target = rng.normal(size=d); target /= np.linalg.norm(target)
    def loss_fn(v):
        v = v / (np.linalg.norm(v) + 1e-8)
        return float(1.0 - (v @ target))  # minimized when v aligns with target
    init = rng.normal(size=d)
    cfg_small = dict(das_spsa_steps=steps, das_spsa_c=0.1, das_spsa_lr=0.5,
                      das_patience=steps, das_early_stop_frac=0.0)
    direction, history = spsa_fit_direction(init, loss_fn, cfg_small, rng=rng)
    assert history[-1] < history[0], f"SPSA loss did not decrease: start={history[0]}, end={history[-1]}"
    return history[0], history[-1]

_d1 = self_test_add_noop()
_d2 = self_test_knockout_zero_noop()
_d3 = self_test_knockout_orthogonalizes()
_h0, _h1 = self_test_spsa_decreases()
print(f"self-test add no-op:            max|delta|   = {_d1:.2e}")
print(f"self-test knockout-zero no-op:  max|delta|   = {_d2:.2e}")
print(f"self-test knockout orthogonal:  residual     = {_d3:.2e}")
print(f"self-test SPSA converges:       loss {_h0:.4f} -> {_h1:.4f}")
print("SELF-TESTS PASSED")


self-test add no-op:            max|delta|   = 0.00e+00
self-test knockout-zero no-op:  max|delta|   = 0.00e+00
self-test knockout orthogonal:  residual     = 3.45e-02
self-test SPSA converges:       loss 1.2254 -> 0.3420
SELF-TESTS PASSED


In [15]:
# This cell is intentionally the ONLY place that assumes real encoded units exist.
# On the real model it should be re-run after GENDER_BANK_READY / ORDER_BANK_READY are
# both True. On the toy harness, the test script below monkeypatches small synthetic
# item banks that DO satisfy the count gates, purely to exercise every code path end to end.

RUN_READY = globals().get("GENDER_BANK_READY", False) and globals().get("ORDER_BANK_READY", False)

if not RUN_READY:
    print("Skipping the full run: item banks are not yet populated past the count gates.")
    print("See Cells 4-5 for exactly what's missing.")
else:
    print("item banks ready -- wire sufficiency_effect/necessity_effect to real encoded batches here,"
          " run the diff-of-means window pass, SPSA fit within that window, alpha* calibration,"
          " controls, bootstrap, and classify_cell/verdict over the resulting cell_report.")
    # NOTE: left as the integration point rather than guessed-at glue code, since the exact
    # encoded-unit plumbing depends on the final GENDER_UNITS/DE_FRAMES/CLAUSE_BANK contents,
    # which are not yet filled in (see Cells 4-5).


Skipping the full run: item banks are not yet populated past the count gates.
See Cells 4-5 for exactly what's missing.


In [16]:
def _mk_cell(task, recipient, cls, R, ci):
    return dict(task=task, recipient=recipient, class_=cls, R=R, R_ci=ci, **{"class": cls})

def force_verdict_branches(cfg):
    def cells(gender_classes, gender_Rs, order_classes, order_Rs):
        out = []
        for recip, c, r in zip(("FR", "ES"), gender_classes, gender_Rs):
            out.append(dict(task="gender", recipient=recip, **{"class": c}, R=r, R_ci=(max(0,r-0.1), r+0.1) if r is not None else (0,0)))
        for recip, c, r in zip(("FR", "ES"), order_classes, order_Rs):
            out.append(dict(task="order", recipient=recip, **{"class": c}, R=r, R_ci=(max(0,r-0.1), r+0.1) if r is not None else (0,0)))
        return out

    v1 = verdict(cells(["TRANSFER","TRANSFER"], [0.8,0.75], ["WALL","WALL"], [0.05,0.1]), cfg)
    v2 = verdict(cells(["TRANSFER","TRANSFER"], [0.8,0.75], ["TRANSFER","TRANSFER"], [0.7,0.8]), cfg)
    v3 = verdict(cells(["WALL","WALL"], [0.05,0.1], ["TRANSFER","TRANSFER"], [0.7,0.8]), cfg)
    v4 = verdict(cells(["TRANSFER","UNUSABLE"], [0.8,None], ["WALL","WALL"], [0.05,0.1]), cfg)
    for name, v in [("inversion",v1),("both_shared",v2),("reversed",v3),("inadequate",v4)]:
        print(name, "->", v[:70])
    assert v1.startswith("INVERSION_SUPPORTED")
    assert v2.startswith("BOTH_SHARED")
    assert v3.startswith("REVERSED")
    assert v4.startswith("INSTRUMENT_INADEQUATE")
    print("VERDICT BRANCHES OK")

force_verdict_branches(CONFIG)

# ---- direct unit tests on the smaller helper functions (not exercised above) ----
def test_classify_cell(cfg):
    assert classify_cell(True, True, True, 0.8, (0.6, 1.0), cfg) == "TRANSFER"
    assert classify_cell(True, True, True, 0.05, (0.0, 0.15), cfg) == "WALL"
    assert classify_cell(True, True, True, 0.35, (0.2, 0.5), cfg) == "AMBIGUOUS"
    assert classify_cell(False, True, True, 0.8, (0.6, 1.0), cfg) == "UNUSABLE"   # sufficiency gate failed
    assert classify_cell(True, False, True, 0.8, (0.6, 1.0), cfg) == "UNUSABLE"   # necessity gate failed
    assert classify_cell(True, True, False, 0.8, (0.6, 1.0), cfg) == "UNUSABLE"   # control gate failed
    print("classify_cell OK")

def test_bootstrap_ci():
    const_vals = [np.array([3.0]) for _ in range(20)]
    mean, lo, hi = bootstrap_ci(const_vals, lambda sample: float(np.mean([s[0] for s in sample])), n_boot=200)
    assert abs(mean - 3.0) < 1e-6 and abs(lo - 3.0) < 1e-6 and abs(hi - 3.0) < 1e-6, (mean, lo, hi)
    print("bootstrap_ci OK (degenerate zero-width case)")

def test_random_and_permutation_directions():
    V = random_directions(d=16, n=50)
    assert V.shape == (50, 16)
    norms = np.linalg.norm(V, axis=1)
    assert np.allclose(norms, 1.0, atol=1e-6)
    rng = np.random.default_rng(0)
    a = rng.normal(size=(10, 16)) + 2.0
    b = rng.normal(size=(10, 16)) - 2.0
    P = permutation_directions(a, b, n=30)
    assert P.shape == (30, 16)
    print("random_directions / permutation_directions OK")

def test_cue_only_and_specificity():
    rng = np.random.default_rng(0)
    x = rng.normal(size=(10, 8)) + 1.0
    y = rng.normal(size=(10, 8)) - 1.0
    cue = cue_only_direction(x, y)
    assert cue.shape == (8,)
    ok, ratio = specificity_2x2(effect_own_task=0.6, effect_cross_task=0.1)
    assert ok and abs(ratio - 6.0) < 1e-9, ratio
    ok2, ratio2 = specificity_2x2(effect_own_task=0.6, effect_cross_task=0.3)
    assert not ok2 and abs(ratio2 - 2.0) < 1e-9
    print("cue_only_direction / specificity_2x2 OK")

def test_transfer_ratio_edge_case():
    assert transfer_ratio(0.4, 0.8) == 0.5
    assert transfer_ratio(0.4, 0.0) is None  # source gate should already have failed upstream
    print("transfer_ratio OK")

test_classify_cell(CONFIG)
test_bootstrap_ci()
test_random_and_permutation_directions()
test_cue_only_and_specificity()
test_transfer_ratio_edge_case()
print("ALL HELPER UNIT TESTS PASSED")


inversion -> INVERSION_SUPPORTED: gender transfers, order does not, CIs separate, c
both_shared -> BOTH_SHARED: both gender and order transfer -- not the inversion.
reversed -> REVERSED: order transfers as much as or more than gender, CIs separate
inadequate -> INSTRUMENT_INADEQUATE: at least one required FR/ES cell is UNUSABLE (g
VERDICT BRANCHES OK
classify_cell OK
bootstrap_ci OK (degenerate zero-width case)
random_directions / permutation_directions OK
cue_only_direction / specificity_2x2 OK
transfer_ratio OK
ALL HELPER UNIT TESTS PASSED
